# 01 非线性方程组的不动点和 Newton 基础

第八章处理的是一个标量方程 $f(x)=0$。本章把问题推广到

$$
F(x)=0,
\qquad x\in\mathbb{R}^n,
\qquad F:\mathbb{R}^n\to\mathbb{R}^n.
$$

变量之间的耦合使问题更复杂：一次更新会同时改变所有分量；Newton 法每步需要解线性方程组；停止准则也要从绝对值变成范数。本节先建立两个基本工具：向量不动点迭代和多元 Newton 法。


In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import fixed_point_system_iteration, newton_system_method


## 向量不动点迭代

若能把方程组改写为

$$
x=G(x),
$$

则可构造

$$
x_{k+1}=G(x_k).
$$

局部收敛条件可以用 Jacobian $G'(\alpha)$ 的范数或谱半径理解：若根附近 $G$ 是压缩映射，迭代通常收敛。实际实现中同时记录步长范数和残差范数。


In [ ]:
def teaching_fixed_point_system(G, x0, residual, tolerance=1e-10, max_iterations=100):
    x_old = np.asarray(x0, dtype=float)
    history = [x_old.copy()]
    residuals = [np.linalg.norm(residual(x_old), ord=2)]
    for k in range(1, max_iterations + 1):
        x_new = np.asarray(G(x_old), dtype=float)
        step = np.linalg.norm(x_new - x_old, ord=2)
        res = np.linalg.norm(residual(x_new), ord=2)
        history.append(x_new.copy())
        residuals.append(res)
        if res <= tolerance or step <= tolerance * max(1.0, np.linalg.norm(x_new, ord=2)):
            return x_new, k, True, np.vstack(history), np.array(residuals)
        x_old = x_new
    return x_old, max_iterations, False, np.vstack(history), np.array(residuals)

G = lambda x: np.array([0.5 * math.cos(x[1]), 0.5 * math.sin(x[0])])
R = lambda x: x - G(x)
solution, iterations, converged, history, residuals = teaching_fixed_point_system(G, [0.3, 0.2], R, tolerance=1e-12)
official = fixed_point_system_iteration(G, [0.3, 0.2], tolerance=1e-12, residual_func=R)

print("teaching solution:", np.array2string(solution, precision=12), "iterations:", iterations, "converged:", converged)
print("official solution:", np.array2string(official.solution, precision=12), "residual:", f"{official.residual_norm:.3e}")
print("first residuals:", np.array2string(residuals[:6], precision=3))


## 多元 Newton 法

对当前点 $x_k$ 作一阶 Taylor 线性化：

$$
F(x_k+s)\approx F(x_k)+J_F(x_k)s.
$$

令右端为零，得到 Newton 步

$$
J_F(x_k)s_k=-F(x_k),
\qquad x_{k+1}=x_k+s_k.
$$

这一步把非线性问题转化为线性方程组。若 Jacobian 非奇异且初值足够接近简单根，Newton 法通常具有局部二次收敛。


In [ ]:
def F(x):
    return np.array([x[0] ** 2 + x[1] ** 2 - 1.0, x[0] - x[1]])


def J(x):
    return np.array([[2.0 * x[0], 2.0 * x[1]], [1.0, -1.0]])

newton = newton_system_method(F, J, [0.8, 0.6], tolerance=1e-12)
expected = np.array([1.0 / math.sqrt(2.0), 1.0 / math.sqrt(2.0)])

print("Newton solution:", np.array2string(newton.solution, precision=15))
print("expected:", np.array2string(expected, precision=15))
print("iterations:", newton.iterations, "residual:", f"{newton.residual_norm:.3e}")
print("residual history:", np.array2string(newton.residual_history, precision=3))


## 奇异 Jacobian 和失败路径

多元 Newton 法比标量 Newton 法更依赖线性代数：若 $J_F(x_k)$ 奇异或病态，线性方程组可能无法稳定求解。后续阻尼 Newton、弦 Newton 和拟 Newton 方法都会围绕“如何让这一步更可靠或更便宜”展开。


In [ ]:
def singular_F(x):
    return np.array([x[0] ** 2 + 1.0, x[1]])


def singular_J(x):
    return np.array([[2.0 * x[0], 0.0], [0.0, 1.0]])

try:
    newton_system_method(singular_F, singular_J, [0.0, 0.0])
except ValueError as exc:
    print("singular Jacobian failure:", exc)


## 本节小结

非线性方程组的核心变化是从标量除法变为矩阵线性求解。向量不动点迭代容易实现，但需要映射在根附近具有压缩性；Newton 法利用 Jacobian 线性化，通常更快，却要求每步求解线性系统，并且可能遭遇奇异或病态 Jacobian。本节的残差历史、步长停止准则和失败路径，是后续阻尼、弦 Newton 和 Broyden 方法的基础。
